# Lab 5: Fine-tuning a coding agent with SFT

## Notebook 4: Deploy to a SageMaker real-time endpoint

This step is optional for the PyTorch Conf flow, the AWS side ends at a model in the
registry, which the edge session consumes. Deploying an endpoint here is useful to
smoke-test the model interactively.

In [ ]:
%load_ext autoreload
%autoreload 2

#### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
import hashlib

from config import BASE_MODEL_ID

base_model_id = BASE_MODEL_ID

# SageMaker caps Model Package Group names at 63 characters. Hash-truncate when
# base_model_id + suffix exceeds it, so every notebook derives the same name.
MAX_MPG_NAME_LENGTH = 63
suffix = "-coding-agent-sft-mpg"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

print(f"Model Package Group: {model_package_group_name}")

### Locate the merged model

Serverless customization writes the merged checkpoint (base weights + LoRA applied) under
`checkpoints/hf_merged/`. That is what we deploy.

In [ ]:
from sagemaker.core import s3
from sagemaker.core.resources import ModelPackage

resp = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime", SortOrder="Descending", MaxResults=1)
assert resp["ModelPackageSummaryList"], "no model packages found - run notebook 2 first"

model_package = ModelPackage.get(resp["ModelPackageSummaryList"][0]["ModelPackageArn"])
merged_model_s3_uri = s3.s3_path_join(
    model_package.inference_specification.containers[0]
    .model_data_source.s3_data_source.s3_uri,
    "checkpoints", "hf_merged") + "/"

print(f"merged model: {merged_model_s3_uri}")

### Resource names

In [ ]:
import hashlib

MAX_NAME = 63


def rname(base, suffix):
    cand = f"{base}{suffix}"
    if len(cand) <= MAX_NAME:
        return cand
    digest = hashlib.sha1(base.encode()).hexdigest()[:6]
    keep = MAX_NAME - len(suffix) - len(digest) - 1
    return f"{base[:keep].rstrip('-')}-{digest}{suffix}"


stem = f"{base_model_id}-coding-agent"
model_name = rname(stem, "-sft-m")
endpoint_config_name = rname(stem, "-sft-cfg")
endpoint_name = rname(stem, "-sft-ep")
ic_name = rname(stem, "-sft-ic")
print(model_name, endpoint_name, ic_name, sep="\n")

### Serving container and environment

In [ ]:
import json

region = sess.boto_region_name
CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

instance_type = "ml.g5.xlarge"
health_check_timeout = 700

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_MAX_MODEL_LEN": json.dumps(4096),
}
print(inference_image)

### Create the model, endpoint config, and endpoint

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import (ContainerDefinition, ModelDataSource,
                                   ProductionVariant, S3ModelDataSource)

try:
    Model.get(model_name)
    print(f"model exists: {model_name}")
except Exception:
    Model.create(
        model_name=model_name,
        primary_container=ContainerDefinition(
            image=inference_image,
            model_data_source=ModelDataSource(
                s3_data_source=S3ModelDataSource(
                    s3_uri=merged_model_s3_uri, s3_data_type="S3Prefix",
                    compression_type="None")),
            environment=env),
        execution_role_arn=role)
    print(f"created model: {model_name}")

try:
    EndpointConfig.get(endpoint_config_name)
    print(f"endpoint config exists: {endpoint_config_name}")
except Exception:
    EndpointConfig.create(
        endpoint_config_name=endpoint_config_name,
        execution_role_arn=role,
        production_variants=[ProductionVariant(
            variant_name="AllTraffic",
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
            routing_config={"routing_strategy": "LEAST_OUTSTANDING_REQUESTS"})])
    print(f"created endpoint config: {endpoint_config_name}")

try:
    endpoint = Endpoint.get(endpoint_name)
    print(f"endpoint exists: {endpoint_name}")
except Exception:
    endpoint = Endpoint.create(endpoint_name=endpoint_name,
                               endpoint_config_name=endpoint_config_name)
    print(f"creating endpoint: {endpoint_name}")

endpoint.wait_for_status("InService")
print("endpoint InService")

### Attach the inference component

In [ ]:
from sagemaker.core.resources import InferenceComponent
from sagemaker.core.shapes import (InferenceComponentComputeResourceRequirements,
                                   InferenceComponentRuntimeConfig,
                                   InferenceComponentSpecification)

try:
    ic = InferenceComponent.get(ic_name)
    print(f"inference component exists: {ic_name}")
except Exception:
    ic = InferenceComponent.create(
        inference_component_name=ic_name,
        endpoint_name=endpoint_name,
        variant_name="AllTraffic",
        specification=InferenceComponentSpecification(
            model_name=model_name,
            compute_resource_requirements=InferenceComponentComputeResourceRequirements(
                min_memory_required_in_mb=1024,
                number_of_accelerator_devices_required=1)),
        runtime_config=InferenceComponentRuntimeConfig(copy_count=1),
        region=region)
    print(f"creating inference component: {ic_name}")

ic.wait_for_status("InService")
print("inference component InService")

### Smoke test on a coding prompt

The prompt is built with `C.build_prompt`, imported rather than pasted, so the served prompt
matches the trained one. `temperature` is 0.0 for a deterministic generation. This is a
serving check, not a measurement, the models are scored only by the pass@1 pipeline in
notebook 3.

In [ ]:
import boto3
from botocore.config import Config

import codeagent as C

smr = boto3.client("sagemaker-runtime", region_name=region,
                   config=Config(read_timeout=300, retries={"total_max_attempts": 3}))


def ask_endpoint(instruction, max_tokens=1024):
    """Invoke the deployed model and return (text, usage)."""
    body = {
        "model_name": ic_name,
        "messages": [{"role": "user",
                      "content": [{"type": "text", "text": C.build_prompt(instruction)}]}],
        "max_tokens": max_tokens,
        "temperature": 0.0,
        "stream": False,
    }
    response = smr.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        ContentType="application/json",
        Body=json.dumps(body),
    )
    payload = json.loads(response["Body"].read())
    return payload["choices"][0]["message"]["content"], payload.get("usage", {})


code_out, usage = ask_endpoint("Write a Python function to find the longest palindromic substring.")
print(code_out)
print("\nusage:", usage)

### Clean up

Delete the **inference component first**, then the endpoint (a 45-second wait between them),
then the config and model. An endpoint bills per instance-hour for as long as it exists.

In [ ]:
import time

for label, fn in [
    ("inference component", lambda: InferenceComponent.get(ic_name).delete()),
    ("endpoint", lambda: Endpoint.get(endpoint_name).delete()),
    ("endpoint config", lambda: EndpointConfig.get(endpoint_config_name).delete()),
    ("model", lambda: Model.get(model_name).delete()),
]:
    try:
        fn()
        print(f"deleted {label}")
    except Exception as e:
        print(f"skip {label}: {type(e).__name__}")
    if label == "inference component":
        time.sleep(45)